In [1]:
import torch
from torch import nn
import matplotlib.pyplot as plt

In [3]:
import torch
print(torch.cuda.is_available())

False


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

c:\Users\computer\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
from datasets import load_dataset

ds = load_dataset("riotu-lab/MURAD")
ds

DatasetDict({
    train: Dataset({
        features: ['word', 'definition', 'ref'],
        num_rows: 96243
    })
})

In [4]:
filtered = [
    (i['word'], i['definition'])
    for i in ds['train']
    if i['word'] is not None and i['definition'] is not None
]

X = [w for w, d in filtered]
y = [d for w, d in filtered]

In [5]:
train_split = int(0.8 * len(X))
X_train, y_train = X[:train_split], y[:train_split]
X_test, y_test = X[train_split:], y[train_split:]

len(X_train), len(y_train), len(X_test), len(y_test)

(76991, 76991, 19248, 19248)

In [7]:
X_train = model.encode(X_train).to("cuda")
y_train = model.encode(y_train).to("cuda")
X_test = model.encode(X_test).to("cuda")
y_test = model.encode(y_test).to("cuda")

ValueError: Modality 'audio' is not supported by this SentenceTransformer model. Supported modalities: text

In [ ]:
torch.manual_seed(0)

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 60),
            nn.ReLU(),
            nn.Linear(60, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, 60),
            nn.ReLU(),
            nn.Linear(60, 3),
        )


    def forward(self, x):
        return self.net(x)


model = Model()

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

losses = []

In [ ]:
epochs = 500

for epoch in range(epochs):
    pred = model(X_test)
    loss = loss_fn(pred, y_test)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if epoch % 50000 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")


print(loss.item())
# ====== Plot ======
plt.figure()
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()